# Mtiririko wa Kazi wa Human-in-the-Loop na Microsoft Agent Framework

## 🎯 Malengo ya Kujifunza

Katika daftari hili, utajifunza jinsi ya kutekeleza mitiririko ya kazi ya **human-in-the-loop** ukitumia Microsoft Agent Framework `RequestInfoExecutor`. Mwelekeo huu wenye nguvu unakuwezesha kusitisha mitiririko ya AI ukusanye maoni ya binadamu, na kufanya maajenti wako kuwa ya mwingiliano na kuwapa wanadamu udhibiti juu ya maamuzi muhimu.

## 🔄 Nini ni Human-in-the-Loop?

**Human-in-the-loop (HITL)** ni mtindo wa kubuni ambapo maajenti wa AI husitisha utekelezaji kuomba maoni ya binadamu kabla ya kuendelea. Hii ni muhimu kwa:

- ✅ **Maamuzi muhimu** - Pata idhini ya binadamu kabla ya kuchukua hatua muhimu
- ✅ **Mazingira yenye kutoeleweka** - Mruhusu wanadamu kufafanua wakati AI haijawiwazi
- ✅ **Mapendeleo ya mtumiaji** - Muulize mtumiaji kuchagua kati ya chaguzi kadhaa
- ✅ **Uzingatiaji na usalama** - Hakikisha usimamizi wa binadamu kwa shughuli zilizo na kanuni
- ✅ **Mazingira ya mwingiliano** - Jenga maajenti wa mazungumzo yanayojibu maoni ya mtumiaji

## 🏗️ Inavyofanya Kazi katika Microsoft Agent Framework

Mfumo huleta vipengele vitatu muhimu kwa HITL:

1. **`RequestInfoExecutor`** - Mtekelezaji maalum anayesitisha mtiririko wa kazi na kutoa `RequestInfoEvent`
2. **`RequestInfoMessage`** - Darasa la msingi la mifano ya maombi ya aina mbalimbali zinazotumwa kwa wanadamu
3. **`RequestResponse`** - Huunganisha majibu ya binadamu na maombi ya awali kwa kutumia `request_id`

**Mwelekeo wa Mtiririko wa Kazi:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 Mfano Wetu: Kuhifadhi Hoteli na Uthibitisho wa Mtumiaji

Tutajenga juu ya mtiririko wa kazi wa masharti kwa kuongeza uthibitisho wa binadamu **kabla** ya kupendekeza maeneo mbadala:

1. Mtumiaji anaomba eneo (mfano, "Paris")
2. `availability_agent` hanakili kama vyumba vinapatikana
3. **Kama hakuna vyumba** → `confirmation_agent` huuliza "Je, ungependa kuona mbadala?"
4. Mtiririko wa kazi **husitishwa** kwa kutumia `RequestInfoExecutor`
5. **Binadamu hujibu** "ndiyo" au "hapana" kupitia ingizo la koni
6. `decision_manager` huongoza kulingana na jibu:
   - **Ndiyo** → Onyesha maeneo mbadala
   - **Hapana** → Ghairi ombi la kuhifadhi
7. Onyesha matokeo ya mwisho

Hii inaonesha jinsi ya kuwapa watumiaji udhibiti juu ya mapendekezo ya maajenti!

---

Tuanze! 🚀


## Hatua ya 1: Ingiza Maktaba Zinazohitajika

Tunaingiza vipengele vya kawaida vya Agent Framework pamoja na **madarasa maalum kwa binadamu katika mzunguko**:
- `RequestInfoExecutor` - Mtendaji anayesimamisha mtiririko wa kazi kwa ajili ya maelezo kutoka kwa binadamu
- `RequestInfoEvent` - Tukio linalotolewa inapokuwa kuna ombi la maelezo kutoka kwa binadamu
- `RequestInfoMessage` - Darasa msingi la mizigo ya maombi iliyowekwa aina
- `RequestResponse` - Linalohusisha majibu ya binadamu na maombi
- `WorkflowOutputEvent` - Tukio la kugundua matokeo ya mtiririko wa kazi


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## Hatua ya 2: Tambua Mifano ya Pydantic kwa Matokeo Yaliyopangwa

Mifano hii inaainisha **mchoro** ambao maajenti watarudisha. Tunahifadhi mifano yote kutoka kwa mtiririko wa kazi wa masharti na kuongeza:

**Mpya kwa Human-in-the-Loop:**
- `HumanFeedbackRequest` - Darasa la chini la `RequestInfoMessage` linaloainisha mzigo wa ombi unaotumwa kwa binadamu
  - Inajumuisha `prompt` (swali la kuuliza) na `destination` (muktadha kuhusu mji usiopatikana)


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## Hatua ya 3: Tengeneza Zana ya Kuweka Hifadhi ya Hoteli

Zana ile ile kutoka kwa mtiririko wa kazi wa masharti - inakagua kama vyumba vinapatikana kwenye sehemu unayoenda.


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## Hatua ya 4: Fafanua Kazi za Hali kwa ajili ya Routing

Tunahitaji **kazi nne za hali** kwa mtiririko wetu wa kazi wa binadamu-ndani-ya-mzunguko:

**Kutoka kwa mtiririko wa kazi wa masharti:**
1. `has_availability_condition` - Huweka njia wakati hoteli ZIPO
2. `no_availability_condition` - Huweka njia wakati hoteli HAZIPO

**Mpya kwa binadamu-ndani-ya-mzunguko:**
3. `user_wants_alternatives_condition` - Huweka njia wakati mtumiaji anasema "ndiyo" kwa mbadala
4. `user_declines_alternatives_condition` - Huweka njia wakati mtumiaji anasema "hapana" kwa mbadala


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## Hatua ya 5: Unda Mtekelezaji wa Meneja wa Uamuzi

Hii ni **kiini cha mtindo wa binadamu-katika-mzunguko**! `DecisionManager` ni `Executor` maalum ambayo:

1. **Inapokea maoni ya binadamu** kupitia vitu vya `RequestResponse`
2. **Inachakata uamuzi wa mtumiaji** (ndiyo/hapana)
3. **Inaelekeza mtiririko wa kazi** kwa kutuma ujumbe kwa mawakala wanaofaa

Vipengele kuu:
- Inatumia `@handler` kama dekoreta kufunua njia kama hatua za mtiririko wa kazi
- Inapokea `RequestResponse[HumanFeedbackRequest, str]` yenye maombi ya awali pamoja na jibu la mtumiaji
- Hutoka na ujumbe rahisi wa "ndiyo" au "hapana" unaochochea kazi zetu za masharti


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## Hatua ya 6: Unda Mtekelezaji wa Onyesho la Kipekee

Mtekelezaji wa onyesho uleule kutoka kwa mtiririko wa kazi wa masharti - hutoa matokeo ya mwisho kama pato la mtiririko wa kazi.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## Hatua ya 7: Pakia Mabadiliko ya Mazingira

Sanidi mteja wa LLM (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## Hatua ya 8: Tengenezaji Wakala wa AI na Watekelezaji

Tunaunda **vipengele sita vya mtiririko wa kazi**:

**Wakala (wamefungwa ndani ya AgentExecutor):**
1. **availability_agent** - Hukagua upatikanaji wa hoteli kwa kutumia chombo
2. **confirmation_agent** - 🆕 Huandaa ombi la uthibitisho wa binadamu
3. **alternative_agent** - Hupendekeza miji mbadala (wakati mtumiaji anasema ndiyo)
4. **booking_agent** - Huhamasisha uhifadhi (wakati vyumba vinapatikana)
5. **cancellation_agent** - 🆕 Hudhibiti ujumbe wa kughairi (wakati mtumiaji anasema hapana)

**Watekelezaji Maalum:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor` ambayo inasimamisha mtiririko wa kazi kwa ajili ya maingizo ya binadamu
7. **decision_manager** - 🆕 Mtekelezaji maalum anayemwelekeza mwelekeo kulingana na majibu ya binadamu (tayari imefafanuliwa hapo juu)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## Hatua ya 9: Jenga Mtiririko wa Kazi kwa Mwanadamu-katika-Mzunguko

Sasa tunatengeneza mchoro wa mtiririko wa kazi na **uwekaji njia wa masharti** ikiwa ni pamoja na njia ya mwanadamu-katika-mzunguko:

**Muundo wa Mtiririko wa Kazi:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**Mikondo Muhimu:**
- `availability_agent → confirmation_agent` (wakati hakuna vyumba)
- `confirmation_agent → prepare_human_request` (badilisha aina)
- `prepare_human_request → request_info_executor` (kisimamizi kwa mwanadamu)
- `request_info_executor → decision_manager` (daima - hutoa RequestResponse)
- `decision_manager → alternative_agent` (wakati mtumiaji asema "ndiyo")
- `decision_manager → cancellation_agent` (wakati mtumiaji asema "hapana")
- `availability_agent → booking_agent` (wakati vyumba vinapatikana)
- Njia zote zinaisha kwa `display_result`


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## Hatua ya 10: Endesha Kesi ya Jaribio 1 - Jiji BILA Upatikanaji (Paris na Uthibitisho wa Binadamu)

Jaribio hili linaonyesha **mzunguko kamili wa binadamu katika mchakato**:

1. Ombi la hoteli huko Paris
2. agenti_wa_Upatikanaji huchunguza → Hakuna vyumba
3. agenti_wa_uthibitisho huunda swali linaloelekezwa kwa binadamu
4. mtekelezaji_wa_ombi_la_info **anasitisha mzunguko wa kazi** na kutoa `RequestInfoEvent`
5. **Programu hutambua tukio na kuomba mtumiaji kwenye consola**
6. Mtumiaji anaandika "ndiyo" au "hapana"
7. Programu inatuma jibu kupitia `send_responses_streaming()`
8. meneja_wa_amuzi anapanga mwelekeo kulingana na jibu
9. Matokeo ya mwisho yanaonyeshwa

**Mfumo Muhimu:**
- Tumia `workflow.run_stream()` kwa mzunguko wa kwanza
- Tumia `workflow.send_responses_streaming(pending_responses)` kwa mizunguko inayofuata
- Sikiliza `RequestInfoEvent` kugundua wakati wa kuhitaji ingizo la binadamu
- Sikiliza `WorkflowOutputEvent` kukamata matokeo ya mwisho


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## Hatua ya 11: Endesha Kesi ya Mtihani 2 - Jiji KWA Upatikanaji (Stockholm - Hakuna Hitaji la Mtu Kuingilia)

Mtihani huu unaonyesha **njia ya moja kwa moja** wakati vyumba vinapatikana:

1. Omba hoteli huko Stockholm
2. agenti wa upatikanaji anachunguza → Vyumba vinapatikana ✅
3. agenti wa uhifadhi anapendekeza uhifadhi
4. display_result inaonyesha uthibitisho
5. **Hakuna hitaji la kuingilia kwa mtu!**

Mchakato wa kazi hupitisha kabisa njia ya mtu katika mzunguko wakati vyumba vinapatikana.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## Muhtasari Muhimu na Mbinu Bora za Binadamu Katika Mzunguko

### ✅ Umejifunza Nini:

#### 1. **Mfumo wa RequestInfoExecutor**
Mfano wa binadamu katika mzunguko katika Microsoft Agent Framework unatumia vipengele vitatu muhimu:
- `RequestInfoExecutor` - Hupunguza mchakato na kutoa matukio
- `RequestInfoMessage` - Darasa la msingi kwa mzigo wa maombi uliotambuliwa (tumia darasa hili!)
- `RequestResponse` - Huhusisha majibu ya binadamu na maombi ya awali

**Ufahamu Muhimu:**
- `RequestInfoExecutor` HAKUKUSANYI maingizo yenyewe - hufunga tu mchakato
- Kode yako lazima isikilize `RequestInfoEvent` na ikusanye maingizo
- Lazima uitaje `send_responses_streaming()` kwa kamusi inayochora `request_id` kwa jibu la mtumiaji

#### 2. **Mfano wa Utekelezaji wa Streaming**
```python
# Iteresheni ya kwanza
stream = workflow.run_stream(initial_request)

# Iteresheni zinazofuata (baada ya maingizo ya binadamu)
stream = workflow.send_responses_streaming(pending_responses)

# Daima chakata matukio
events = [event async for event in stream]
```

#### 3. **Muundo Unaotegemea Matukio**
Sikiliza matukio maalum kudhibiti mchakato:
- `RequestInfoEvent` - Mahitaji ya maingizo ya binadamu (mchakato umeachwa)
- `WorkflowOutputEvent` - Matokeo ya mwisho yapatikana (mchakato umekamilika)
- `WorkflowStatusEvent` - Mabadiliko ya hali (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS, nk.)

#### 4. **Watekelezaji Maalum kwa @handler**
`DecisionManager` inaonyesha jinsi ya kuunda watekelezaji ambao:
- Wanatumia chakelezo cha `@handler` kufunua njia za hatua za mchakato
- Wanapokea ujumbe ulioainishwa (mfano, `RequestResponse[HumanFeedbackRequest, str]`)
- Wanapanga mchakato kwa kutuma ujumbe kwa watekelezaji wengine
- Wanapokea muktadha kupitia `WorkflowContext`

#### 5. **Kuelekeza Kwa Masharti kwa Maamuzi ya Binadamu**
Unaweza kuunda kazi za masharti zinazotathmini majibu ya binadamu:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 Matumizi Halisi:

1. **Mifumo ya Idhini**
   - Pata idhini ya meneja kabla ya kushughulikia ripoti za gharama
   - Inahitaji uhakiki wa binadamu kabla ya kutuma barua pepe za mt automate
   - Thibitisha miamala ya thamani kubwa kabla ya utekelezaji

2. **Udhibiti wa Maudhui**
   - Taja maudhui yanayoshukiwa kwa uhakiki wa binadamu
   - Waulize watendaji kufanya uamuzi wa mwisho kwa kesi ngumu
   - Ongezea kwa binadamu wakati uhakika wa AI ni mdogo

3. **Huduma kwa Wateja**
   - Ruhusu AI kushughulikia maswali ya kawaida kiotomatiki
   - Ongezea masuala changamano kwa maajenti binadamu
   - Muulize mteja ikiwa anataka kuzungumza na binadamu

4. **Usindikaji wa Data**
   - Muulize binadamu kutatua ingizo la data ambalo halijulikani
   - Thibitisha tafsiri za AI za nyaraka zisizoeleweka
   - Mruhusu watumiaji kuchagua kati ya tafsiri mbalimbali sahihi

5. **Mifumo Muhimu kwa Usalama**
   - Inahitaji uthibitisho wa binadamu kabla ya hatua zisizoweza kurudishwa
   - Pata idhini kabla ya kupokea data nyeti
   - Thibitisha maamuzi katika sekta zinazodhibitiwa (huduma za afya, fedha)

6. **Maajenti Waingilie**
   - Tengeneza roboti wa mazungumzo wanaouliza maswali ya kuendelea
   - Unda wasaidizi wanaowaongoza watumiaji kupitia michakato migumu
   - Buni maajenti wanaoshirikiana na binadamu hatua kwa hatua

### 🔄 Ulinganisho: Kwa vs Bila Binadamu Katika Mzunguko

| Kipengele | Mchakato wa Masharti | Mchakato wa Binadamu Katika Mzunguko |
|---------|---------------------|---------------------------|
| **Utekelezaji** | `workflow.run()` moja | Mzunguko na `run_stream()` + `send_responses_streaming()` |
| **Maingizo ya Mtumiaji** | Hakuna (kiotomatiki kabisa) | Vitisho vya maingizo kupitia `input()` au UI |
| **Vipengele** | Madereva + Watekelezaji | + RequestInfoExecutor + DecisionManager |
| **Matukio** | AgentExecutorResponse tu | RequestInfoEvent, WorkflowOutputEvent, nk. |
| **Kusimamisha** | Hakuna kusimamisha | Mchakato unasimamishwa kwenye RequestInfoExecutor |
| **Udhibiti wa Binadamu** | Hakuna udhibiti wa binadamu | Wanadamu hufanya maamuzi muhimu |
| **Matumizi** | Ujia | Ushirikiano & ukaguzi |

### 🚀 Mifano ya Juu:

#### Sehemu Nyingi za Maamuzi ya Binadamu
Unaweza kuwa na nodi nyingi za `RequestInfoExecutor` katika mchakato mmoja:
```python
.add_edge(agent1, request_info_1)  # Uamuzi wa mwanadamu wa kwanza
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # Uamuzi wa mwanadamu wa pili
.add_edge(decision_manager_2, final_agent)
```

#### Kusimamia Muda wa Kusubiri
Tekeleza muda wa kusubiri kwa majibu ya binadamu:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # Chagua chaguo salama kama msingi
```

#### Ushirikiano wa UI Tajiri
Badala ya `input()`, shiriki na UI za wavuti, Slack, Teams, nk.:
```python
if isinstance(event, RequestInfoEvent):
    # Tuma arifa kwenye njia ya mtumiaji anayopendelea
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### Binadamu Katika Mzunguko Kwa Masharti
Uliza maingizo ya binadamu tu katika hali maalum:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # Tuma kwa binadamu tu kama kujiamini ni kidogo au thamani ni kubwa
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ Mbinu Bora:

1. **Daima Tumia Darasa la RequestInfoMessage**
   - Hutoa usalama wa aina na uthibitisho
   - Huruhusu muktadha tajiri kwa ajili ya UI
   - Huweka wazi nia ya aina ya kila ombi

2. **Tumia Masharti Yanayofafanua**
   - Jumuisha muktadha wa unachouliza
   - Eleza madhara ya kila chaguo
   - Weka maswali kuwa rahisi na wazi

3. **Shughulikia Maingizo Yasiyotarajiwa**
   - Thibitisha majibu ya mtumiaji
   - Toa chaguo mbadala kwa maingizo yasiyo sahihi
   - Toa ujumbe wa makosa ulio wazi

4. **Fuata Request ID**
   - Tumia uhusiano kati ya request_id na majibu
   - Usijaribu kudhibiti hali kwa mkono

5. **Tengeneza kwa ajili ya Kutokuwa na Mzigo**
   - Usizuie nyuzi unazosubiri maingizo
   - Tumia mifano ya async kote
   - Saidia matukio ya mchakato yanayofanana

### 📚 Dhana Zinazohusiana:

- **Middleware ya Agent** - Kata simu za maajenti (daftari lililopita)
- **Usimamizi wa Hali ya Workflow** - Hifadhi hali ya mchakato kati ya mizunguko
- **Ushirikiano wa Maajenti Wengi** - Changanya binadamu katika mzunguko na timu za maajenti
- **Miundo Yanayotegemea Matukio** - Unda mifumo inayojibu matukio

---

### 🎓 Hongera!

Umejifunza workflows za binadamu katika mzunguko kwa Microsoft Agent Framework! Sasa unajua jinsi ya:
- ✅ Kusimamisha workflows kukusanya maingizo ya binadamu
- ✅ Kutumia RequestInfoExecutor na RequestInfoMessage
- ✅ Kusimamia utekelezaji wa streaming kwa matukio
- ✅ Kuunda watekelezaji maalum kwa @handler
- ✅ Kuelekeza workflows kwa maamuzi ya binadamu
- ✅ Kujenga maajenti wa AI wa kiingilizi wanaoshirikiana na binadamu

**Huu ni mfano muhimu wa kujenga mifumo ya AI ya kuaminika, inayoweza kudhibitiwa!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
